In [1]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 12.7 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [catboost]3/4 [catboost]
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import pandas as pd
import numpy as np
import joblib
import warnings

# Suppress specific warnings, especially from scikit-learn
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')
warnings.filterwarnings("ignore", category=FutureWarning, module='sklearn')

# --- Configuration ---
# === INPUTS: MUST BE SET BY USER ===

# 1. Name of the model you want to use for prediction (e.g., 'RandomForest', 'XGBoost')
#    This name should match the prefix of your saved pipeline file.
MODEL_NAME_FOR_PREDICTION = 'RandomForest' # <<< CHOOSE ONE MODEL HERE

# 2. Directory where your saved model pipelines are located
MODEL_DIR = "mic_activity_prediction_study_v4/models/" # <<< SET PATH (e.g., where 'RandomForest_best_pipeline_v1.pkl' is)

# 3. Filename suffix for your saved pipelines
PIPELINE_SUFFIX = "_best_model.pkl" # <<< SET SUFFIX (e.g., '_best_pipeline_v1.pkl')

# 4. Path to the file containing the list of selected features used by this model
#    This is critical for ensuring your new data has the correct features in the correct order.
FEATURES_FILE = "rfe_selected_features.csv" # <<< SET PATH

# 5. Path to the new dataset CSV file you want to predict on
#    This file should contain all the features listed in your FEATURES_FILE.
NEW_DATA_PATH = "Descriptors_Dr.Khalid_Compounds.csv" # <<< SET PATH to your new data

# 6. Optional: Name of a unique identifier column in your new data (e.g., 'Molecule_ID', 'mol_id')
#    If you don't have one, set to None. This column will be included in the output.
ID_COLUMN = 'mol_id' # <<< SET ID COLUMN NAME or None

# 7. Output directory for saving the prediction results
PREDICTION_OUTPUT_DIR = "Model_Predictions" # <<< SET OUTPUT DIRECTORY

# 8. Output format for predictions (currently only 'csv' is implemented)
OUTPUT_FORMAT = 'csv'

# ===================================

# Set a random state for reproducibility if any underlying operations are random
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Create Output Directory ---
print(f"Creating output directory: {PREDICTION_OUTPUT_DIR}")
os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)
print(f"  Directory created/exists: {PREDICTION_OUTPUT_DIR}")

'''
# --- Helper Function to Load Features ---
def load_features_from_file(filepath):
    """Loads feature names from a text file, one feature per line."""
    print(f"\nLoading selected features from {filepath}...")
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f:
                features = [line.strip() for line in f if line.strip()]
            if not features:
                print("  Error: Features file is empty.")
                return None
            print(f"  Successfully loaded {len(features)} features.")
            return features
        except Exception as e:
            print(f"  Error loading features from file: {e}.")
            return None
    else:
        print(f"  Error: Features file not found at '{filepath}'.")
        return None

# --- Load Selected Features ---
selected_features = load_features_from_file(FEATURES_FILE)
if selected_features is None:
    print("Exiting: Failed to load selected features.")
    exit()
'''
# Above commented out section if feature list available in txt file
# --- Load Selected Features ---
print(f"\nLoading selected features from {FEATURES_FILE}...")
try:
    # Read the CSV and get the feature names from the first column
    features_df = pd.read_csv(FEATURES_FILE)
    selected_features = features_df.iloc[:, 0].tolist() # Gets all rows from the first column
    
    if not selected_features:
        print("  Error: The features file is empty.")
        exit()
    
    print(f"  Successfully loaded {len(selected_features)} features.")

except FileNotFoundError:
    print(f"  Error: Features file not found at '{FEATURES_FILE}'.")
    exit()
except Exception as e:
    print(f"  Error loading features from {FEATURES_FILE}: {e}.")
    exit()
# --- Load New Data for Prediction ---
print(f"\nLoading new data from {NEW_DATA_PATH}...")
new_df = None # Initialize DataFrame
X_new = None  # Initialize features for prediction
ids = None    # Initialize ID column

try:
    new_df = pd.read_csv(NEW_DATA_PATH)
    print(f"  New data shape: {new_df.shape}")

    # Check if ID column exists and store it
    if ID_COLUMN and ID_COLUMN in new_df.columns:
        ids = new_df[ID_COLUMN].copy()
        print(f"  Using ID column: '{ID_COLUMN}' from new data.")
    elif ID_COLUMN: # If ID_COLUMN is specified but not found
        print(f"Warning: Specified ID column '{ID_COLUMN}' not found in the new data. Predictions will not include this ID.")

    # Verify all selected features are present in the new data
    missing_features_in_new_data = [f for f in selected_features if f not in new_df.columns]
    if missing_features_in_new_data:
        print(f"Error: The following selected features are missing in your new data file ({NEW_DATA_PATH}):")
        for feat in missing_features_in_new_data:
            print(f"  - {feat}")
        print("Please ensure your new data contains all the features specified in your FEATURES_FILE.")
        exit() # Terminate if critical features are missing

    # Prepare data subset using selected features
    X_new = new_df[selected_features].copy()
    print(f"  Prepared new data for prediction with {X_new.shape[1]} features.")

except FileNotFoundError:
    print(f"Error: New data file not found at '{NEW_DATA_PATH}'. Check path.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred while loading or preparing new data: {e}")
    exit()


# --- Make Predictions with Loaded Model ---
print("\n" + "="*60)
print(f"--- Starting Prediction with Model: {MODEL_NAME_FOR_PREDICTION} ---")
print("="*60)

all_predictions = {} # Dictionary to store prediction results

# Add IDs to the prediction results if available
if ids is not None:
    all_predictions[ID_COLUMN] = ids

# Add original features to the predictions DataFrame for context (optional)
# This can make the output file very wide if you have many features
# for feature_col in X_new.columns:
#     all_predictions[f"feature_{feature_col}"] = X_new[feature_col]


# --- Load Model Pipeline ---
pipeline_path = os.path.join(MODEL_DIR, f"{MODEL_NAME_FOR_PREDICTION}{PIPELINE_SUFFIX}")
print(f"  Attempting to load pipeline from: {pipeline_path}")

pipeline = None
try:
    pipeline = joblib.load(pipeline_path)
    print(f"  Pipeline '{MODEL_NAME_FOR_PREDICTION}' loaded successfully.")
except FileNotFoundError:
    print(f"Error: Pipeline file not found for '{MODEL_NAME_FOR_PREDICTION}'. Check MODEL_DIR and PIPELINE_SUFFIX.")
    print("Exiting prediction process.")
    exit()
except Exception as e:
    print(f"Error loading pipeline for '{MODEL_NAME_FOR_PREDICTION}': {e}.")
    print("Please ensure the model was saved correctly and all necessary libraries are installed.")
    print("Exiting prediction process.")
    exit()

# --- Perform Predictions ---
if pipeline is not None:
    try:
        # Predict probabilities if the model supports it
        if hasattr(pipeline, 'predict_proba'):
            print("  Predicting probabilities...")
            predictions_proba = pipeline.predict_proba(X_new)

            # Store probabilities for each class (e.g., class 0 and class 1)
            # Assuming binary classification where classes are typically 0 and 1
            if len(pipeline.classes_) == 2:
                all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_proba_class_{pipeline.classes_[0]}"] = predictions_proba[:, 0]
                all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_proba_class_{pipeline.classes_[1]}"] = predictions_proba[:, 1]
                # Often, we only care about the probability of the positive class (e.g., class 1)
                all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_probability"] = predictions_proba[:, 1]
            else: # Handle multi-class case if needed, or simplify for binary
                for i, class_name in enumerate(pipeline.classes_):
                    all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_proba_class_{class_name}"] = predictions_proba[:, i]

            print("  Predicting class labels...")
            predictions_labels = pipeline.predict(X_new)
            all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_predicted_label"] = predictions_labels

        # Otherwise, just predict class labels
        elif hasattr(pipeline, 'predict'):
            print("  Model does not have 'predict_proba', predicting class labels only...")
            predictions_labels = pipeline.predict(X_new)
            all_predictions[f"{MODEL_NAME_FOR_PREDICTION}_predicted_label"] = predictions_labels
        else:
            print("  Model does not have 'predict' or 'predict_proba' method. Cannot make predictions.")
            print("No predictions generated.")

    except Exception as e:
        print(f"An error occurred during prediction for '{MODEL_NAME_FOR_PREDICTION}': {e}.")
        print("No predictions generated due to this error.")


# --- Save Predictions to Output File ---
if all_predictions and any(col.startswith(MODEL_NAME_FOR_PREDICTION) for col in all_predictions.keys()):
    predictions_df = pd.DataFrame(all_predictions)

    # Construct a dynamic output filename
    output_filename = f"{MODEL_NAME_FOR_PREDICTION}_predictions_on_external_data.{OUTPUT_FORMAT}"
    output_path = os.path.join(PREDICTION_OUTPUT_DIR, output_filename)

    print(f"\nSaving predictions to: {output_path}...")

    try:
        if OUTPUT_FORMAT.lower() == 'csv':
            predictions_df.to_csv(output_path, index=False)
            print("Predictions saved successfully!")
        else:
            print(f"Error: Unsupported output format '{OUTPUT_FORMAT}'. Predictions not saved.")

    except Exception as e:
        print(f"Error saving predictions: {e}")
else:
    print("\nNo predictions were generated. Output file not created.")

print("\n" + "="*60)
print("--- Prediction Process Finished ---")
print("="*60)

Creating output directory: Model_Predictions
  Directory created/exists: Model_Predictions

Loading selected features from rfe_selected_features.csv...
  Successfully loaded 20 features.

Loading new data from Descriptors_Dr.Khalid_Compounds.csv...
  New data shape: (4105, 17537)
  Prepared new data for prediction with 20 features.

--- Starting Prediction with Model: RandomForest ---
  Attempting to load pipeline from: mic_activity_prediction_study_v4/models/RandomForest_best_model.pkl
  Pipeline 'RandomForest' loaded successfully.
  Predicting probabilities...
  Predicting class labels...

Saving predictions to: Model_Predictions/RandomForest_predictions_on_external_data.csv...
Predictions saved successfully!

--- Prediction Process Finished ---
